# 01 · Tokenize the pretraining corpus

Streams the three sources, tokenizes them with the project tokenizer, writes uint16 shards, and uploads every finished shard to a **private** Hugging Face dataset repo.

| Source | Train tokens | Validation tokens |
|---|---:|---:|
| Turkish (FineWeb2-HQ `tur_Latn`) | 230M | 10M |
| Python (codeparrot-clean) | 700M | 10M |
| English (FineWeb-Edu `sample-10BT`) | 1.18B | 10M |

**Runtime:** *Select Kernel → Colab → New Colab Server → **CPU***. Tokenization uses no GPU.

**Resumable:** every shard is uploaded as soon as it is written, together with a per-source manifest. If the session ends, reconnect and run all cells again; finished shards are skipped.

**Measure first:** watch the `tokens/s` printed after the first Turkish shard and estimate the total time before letting it run.

## 1. Hardware

In [ ]:
import os
import shutil

meminfo = dict(line.split(":", 1) for line in open("/proc/meminfo"))
print("CPU cores:", os.cpu_count())
print("RAM GiB:  ", round(int(meminfo["MemTotal"].split()[0]) / 1024**2, 1))
print("Disk free GiB:", round(shutil.disk_usage("/content").free / 1024**3, 1))

## 2. Get the code

Set `REPO_URL` to your GitHub repository.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Tbt-code/GLM-Ladder-from-Scratch.git"  # <- edit
BRANCH = "main"
REPO_DIR = Path("/content/glm-ladder-from-scratch")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--quiet", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--quiet", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(REPO_DIR / "requirements.txt")], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("commit:", commit)

## 3. Run the data tests

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_shards.py", "tests/test_tokenizer.py", "tests/test_data_sources.py"],
    capture_output=True, text=True,
)
print(result.stdout[-2000:], result.stderr[-1000:])
assert result.returncode == 0, "tests failed - fix before tokenizing"

## 4. Configuration

`REPO_ID` is your private dataset repo. The tokenizer hash is checked against the receipt in git, so the shards are guaranteed to match the committed tokenizer.

In [ ]:
import hashlib
import json

REPO_ID = "<Tbt-AI>/glm-ladder-data"  # <- edit
TOKENIZER_PATH = REPO_DIR / "tokenizer" / "glm-ladder-16k.json"
WORKDIR = Path("/content/shards")
SHARD_TOKENS = 100_000_000      # 200 MB per shard as uint16
VAL_TOKENS = 10_000_000
TRAIN_TOKENS = {                # order matters: smallest first, to measure speed early
    "turkish": 230_000_000,
    "code": 700_000_000,
    "english": 1_180_000_000,
}

receipt = json.loads((REPO_DIR / "artifacts" / "receipts" / "tokenizer-16k.json").read_text())
actual = hashlib.sha256(TOKENIZER_PATH.read_bytes()).hexdigest()
assert actual == receipt["sha256"], "tokenizer file does not match its receipt"
print("tokenizer sha256 ok:", actual[:16], "...")

## 5. Log in to Hugging Face

The token is typed into a hidden prompt and kept only in this session's memory. Use the fine-grained token that can write to `glm-ladder-data` only. Never paste it into a cell.

In [ ]:
from getpass import getpass

from huggingface_hub import HfApi

api = HfApi(token=getpass("Hugging Face write token: "))
print("logged in as:", api.whoami()["name"])
info = api.repo_info(REPO_ID, repo_type="dataset")
assert info.private, "the dataset repo must be private"
print("repo ok and private:", REPO_ID)

## 6. Tokenize and upload

Each source logs one line per finished shard with the running `tokens/s`. Stopping this cell (or losing the session) is safe: rerun all cells and it continues after the last uploaded shard.

In [ ]:
import time

from glm_ladder.corpus import tokenize_source

overall = time.perf_counter()
for source, train_tokens in TRAIN_TOKENS.items():
    tokenize_source(
        api, REPO_ID, source,
        tokenizer_path=TOKENIZER_PATH, train_tokens=train_tokens, val_tokens=VAL_TOKENS,
        shard_tokens=SHARD_TOKENS, workdir=WORKDIR,
    )
print(f"all sources finished in {(time.perf_counter() - overall) / 3600:.2f} h")

## 7. Summary receipt

Copy the printed JSON into `artifacts/receipts/corpus-manifest.json` in VS Code and commit it. The full per-shard manifests stay in the dataset repo.

In [ ]:
from glm_ladder.corpus import load_manifest

summary = {"schema_version": "1.0", "kind": "pretraining_corpus", "repo_id": REPO_ID,
           "commit": commit, "tokenizer_sha256": actual, "sources": {}}
for source in TRAIN_TOKENS:
    manifest = load_manifest(api, REPO_ID, source)
    shards = manifest["shards"]
    summary["sources"][source] = {
        "dataset": manifest.get("dataset"),
        "train_tokens": sum(s["tokens"] for s in shards if s["name"].startswith("train-")),
        "val_tokens": sum(s["tokens"] for s in shards if s["name"].startswith("val-")),
        "shards": len(shards),
        "documents": shards[-1]["documents_end"] if shards else 0,
        "skipped_documents": sum(s["skipped_documents"] for s in shards),
    }
summary["train_tokens_total"] = sum(s["train_tokens"] for s in summary["sources"].values())
print(json.dumps(summary, indent=2))